# Inspect FAISS retrieval for an eligibility question

This notebook searches the persisted local FAISS store and displays the exact chunks that would be supplied to the answer layer.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from src.embeddings.build_faiss_index import MODEL_DIR, MODEL_ID, VECTOR_STORE_DIR

index = faiss.read_index(str(VECTOR_STORE_DIR / 'index.faiss'))
chunks = json.loads((VECTOR_STORE_DIR / 'chunks.json').read_text(encoding='utf-8'))
model = SentenceTransformer(MODEL_ID, cache_folder=str(MODEL_DIR), local_files_only=True)
print(f'Loaded {index.ntotal} vectors with {index.d} dimensions.')

d:\Projects\Scheme Bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3504.94it/s]

Loaded 274 vectors with 768 dimensions.


In [2]:
# Change this value to inspect another user question.
QUESTION = 'Am I eligible to apply for the Annal Ambedkar Business Champions Scheme?'
TOP_SCHEMES = 5
CHUNK_CANDIDATES = 30

query = 'Represent this sentence for searching relevant passages: ' + QUESTION
query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
scores, indices = index.search(query_vector, CHUNK_CANDIDATES)

# Keep only the highest scoring chunk for each scheme.
results, seen_source_files = [], set()
for chunk_rank, (score, chunk_index) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = chunks[int(chunk_index)]
    source_file = chunk['metadata']['source_file']
    if source_file in seen_source_files:
        continue
    seen_source_files.add(source_file)
    results.append({'scheme_rank': len(results) + 1, 'first_chunk_rank': chunk_rank, 'score': float(score), 'chunk': chunk})
    if len(results) == TOP_SCHEMES:
        break

print(f'Question: {QUESTION}')
print(f'Retrieved {len(results)} distinct schemes.')

Question: Am I eligible to apply for the Annal Ambedkar Business Champions Scheme?
Retrieved 5 distinct schemes.


In [3]:
from textwrap import fill

for result in results:
    chunk = result['chunk']
    metadata = chunk['metadata']
    print('=' * 100)
    print(f"Scheme rank: {result['scheme_rank']} | cosine score: {result['score']:.4f} | first chunk rank: {result['first_chunk_rank']}")
    print(f"Scheme: {metadata.get('scheme_name')}")
    print(f"Source: {metadata.get('source_file')} | section: {metadata.get('section')} | chunk: {chunk.get('chunk_id')}")
    print('-' * 100)
    print(fill(chunk['text'], width=110))
    print()

Scheme rank: 1 | cosine score: 0.7288 | first chunk rank: 1
Scheme: Annal Ambedkar Business Champions Scheme
Source: aabcs.txt | section: General | chunk: aabcs.txt_recursive_1
----------------------------------------------------------------------------------------------------
Annal Ambedkar Business Champions SchemeAre you sure you want to  ?  Business Champions Scheme (AABCS)â€​ was
launched by the Department of Micro, Small and Medium Enterprises, Government of Tamil Nadu. The Government of
Tamil Nadu noted that the   of persons from Scheduled Caste and Scheduled Tribe communities who avail of
various entrepreneurship promotion subsidies from the Government is very low. In order to promote the economic
development of SC/ST entrepreneurs, this scheme has been launched by the Government of Tamil Nadu. The new
scheme will provide a 35% capital subsidy and offer a 6% interest subvention for loans to procure machinery
and equipment. This scheme shall be implemented by the Micro, Small an

For the AABCS question, the first result should be the Tamil Nadu scheme and its eligibility chunk should mention residence, SC/ST status, age, ownership, and eligible business types. The answer layer should only summarize this retrieved evidence and ask for missing user facts.